# Notebook 04: Integración de LLM Local y Explicabilidad SHAP
Este notebook implementa el puente de conexión entre el modelo predictivo XGBoost (Persona 1) y el modelo de lenguaje de Ollama (Persona 2). 

**Flujo:**
1. Carga del preprocesador y modelo entrenado.
2. Carga de datos de prueba y selección de un cliente con alto riesgo.
3. Cálculo local de valores SHAP y traducción a nombres legibles de negocio.
4. Construcción del prompt estructurado.
5. Consulta local en Ollama (Mistral) para generar un reporte de retención.

In [ ]:
# 1. Imports y configuración
import os
import sys
import joblib
import pandas as pd
import shap
import ollama

# Asegurar imports de src
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.agents.ollama_explain import clean_feature_name, get_shap_explanation, generate_llm_explanation

checkpoints_dir = "../models/checkpoints"
test_data_path = "../data/processed/test.csv"

print("Módulos importados correctamente.")

In [ ]:
# 2. Cargar modelos y datos de prueba
preprocesador = joblib.load(os.path.join(checkpoints_dir, "preprocessor.joblib"))
model = joblib.load(os.path.join(checkpoints_dir, "xgb_model.joblib"))
df_test = pd.read_csv(test_data_path)

print(f"Conjunto de prueba cargado: {df_test.shape[0]} registros.")
print("Modelos cargados desde checkpoints.")

In [ ]:
# 3. Seleccionar un cliente de prueba con alto riesgo de fuga
X_test = df_test.drop(columns=["Churn"])
y_test = df_test["Churn"]

X_test_trans = preprocesador.transform(X_test)
probabilidades = model.predict_proba(X_test_trans)[:, 1]

# Buscar un cliente que tenga alta probabilidad de fuga
df_res = pd.DataFrame({
    "index": df_test.index,
    "Probabilidad": probabilidades,
    "Real": y_test
})

clientes_riesgo = df_res[df_res["Probabilidad"] > 0.65].sort_values(by="Probabilidad", ascending=False)
print("Top clientes con mayor riesgo detectado:")
print(clientes_riesgo.head())

# Seleccionamos el cliente con mayor riesgo
target_idx = clientes_riesgo.iloc[0]["index"].astype(int)
client_row = X_test.iloc[[target_idx]]
client_prob = probabilidades[target_idx]
client_real = y_test.iloc[target_idx]

print(f"\nCliente Seleccionado (Index: {target_idx}):")
print(f"Probabilidad de Churn predicha: {client_prob * 100:.2f}%")
print(f"Churn Real: {'Sí' if client_real == 1 else 'No'}")

In [ ]:
# 4. Explicabilidad SHAP con Nombres Decodificados
top_factors = get_shap_explanation(model, preprocesador, client_row)

print("Factores de impacto SHAP para este cliente:")
for factor, val in top_factors.items():
    print(f"{factor:<35} : {val:+.4f}")

In [ ]:
# 5. Generar reporte con Ollama local (Mistral)
llm_output = generate_llm_explanation(top_factors, client_prob, model_name="mistral")
print("\n--- REPORTE EXECUTIVO DE IA ---")
print(llm_output)